# Fit e Transform

- fit é pra treinamento do modelo 
- transform é pra reutilizar os dados que foram aprendidos pra fazer algo, pegar dados novos, aplicar o que foi aprendido e transformar em alguma coisa nova 

In [3]:
# pandas oara montar tabelas (DataFrame)
import pandas as pd 

# para salvar e carregar o pipeline de treinamento 
import joblib

# criar pasta/arquivo
from pathlib import Path

# ferramentas de sklearn
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder # transformar a string para numérico
from sklearn.compose import ColumnTransformer # aplica a função de transformação em cada coluna 
from sklearn.pipeline import Pipeline # montagem da pipeline

In [5]:
class MiniFeatureEngineer: 
    def __init__(self, model_dir="models"):
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(parents=True, exists_ok=True) # se não existir, ele cria, se existir, nao da erro 
        self.path = self.model_dir # mini_transform.joblib 

        # serão preenchidos com fit() ou load()
        self.pipeline = None
        self.feature_names = None # colunas

    def fit(self, df, num_cols, cat_cols):
        # pré-processamento que irá aplicar o column transformer nas colunas que eu passar
        pre = ColumnTransformer(
            transformers = [
                ("num", MinMaxScaler(), num_cols),
                # transforma em 0 e 1 
                ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, categories=cat_cols)) # ignora categorias novas pra nao dar erro
            ]
        )

        # vai passar cada uma das funções do pré-processamento para cada coluna, e depois vai juntar tudo em um pipeline
        self.pipeline = Pipeline(steps=[("preprocessor", pre)])

        # fit - vai aprender o min/max e categorias 
        self.pipeline.fit(df) # vai aprender a transformar os dados de acordo com o que tem no df

        # pega as categorias do one hot encoder 
        cat = self.pipeline.named_steps["preprocessor"].named_transformers_["cat"]
        self.features_names = num_cols + cat.get_feature_names_out(cat_cols).tolist()

    # só transforma os dados, e depois eu decido o que fazer, mandar pra treinamento ou descartar 
    def transform(self, df):
        if self.pipeline is None:
            self.load()

        # transform - aplica as regras aprendidas no fit sem reaprender nada, só transforma os dados novos de acordo com o que foi aprendido no fit
        data = self.pipeline.transform(df)
        return pd.DataFrame(data, columns=self.feature_names)    

    def save(self):
        # salva tudo que foi aprendido no fit no .joblib
        joblib.dump({"pipeline": self.pipeline, "feature_names": self.features_names}, self.path)

    def load(self):
        state = joblib.load(self.path) # carrega o arquivo .joblib
        self.pipeline = state["pipeline"] # pega o pipeline que foi salvo
        self.feature_names = state["feature_names"] # pega os nomes das features que foram salvas
